In [1]:
# ######### used this part for fixing problems running on ARC #
import os
os.environ['HF_HOME'] = '/projects/cesca-cv/ishtiaque/models/'
os.environ['HF_HUB_CACHE'] = '/projects/cesca-cv/ishtiaque/models/'
os.environ['XDG_CACHE_HOME'] = '/projects/cesca-cv/ishtiaque/models/'
os.environ['NB_USER'] = 'ishtiahmed'
os.environ['TRANSFORMERS_CACHE'] = '/projects/cesca-cv/ishtiaque/models/'
os.environ['HF_DATASETS_CACHE'] = '/projects/cesca-cv/ishtiaque/models/'

In [2]:
from qwen_vl_utils import process_vision_info
from transformers import AutoTokenizer, AutoProcessor, AutoModelForCausalLM


import torch
import os 
import json
import random
from tqdm import tqdm
from collections import Counter 

import numpy as np
import torch
import torchvision.transforms as T



/projects/abbott_lab/Users/ishtiaque/env/llava_ovision_1_5_instruct_8B/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/projects/abbott_lab/Users/ishtiaque/env/llava_ovision_1_5_instruct_8B/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
model_path = "lmms-lab/LLaVA-OneVision-1.5-8B-Instruct"

# default: Load the model on the available device(s)
model = AutoModelForCausalLM.from_pretrained(
    model_path, torch_dtype="auto", device_map="auto", trust_remote_code=True
)

# default processer
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)




A new version of the following files was downloaded from https://huggingface.co/lmms-lab/LLaVA-OneVision-1.5-8B-Instruct:
- configuration_llavaonevision1_5.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/lmms-lab/LLaVA-OneVision-1.5-8B-Instruct:
- modeling_llavaonevision1_5.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [01:13<00:00, 18.49s/it]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow process

In [4]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg",
            },
            {"type": "text", "text": "Describe this image."},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=1024)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

["The image depicts a serene beach scene during what appears to be either sunrise or sunset, given the warm, golden light bathing the surroundings. The main subjects of the image are a woman and her dog.\n\n**Objects Present:**\n1. **Woman:** She is seated on the sandy beach, facing towards the right side of the image. She has long hair and is wearing a plaid shirt with rolled-up sleeves. Her legs are crossed, and she is smiling while looking at the dog.\n2. **Dog:** A large, tan-colored dog (possibly a Labrador Retriever) is sitting next to the woman. The dog is wearing a colorful harness and is reaching out its paw towards the woman's hand in a friendly gesture.\n3. **Beach:** The sandy beach occupies most of the lower part of the image. The sand appears smooth with some footprints and disturbances from the dog.\n4. **Ocean:** In the background, there is a calm ocean with gentle waves approaching the shore. The water is relatively clear, and the horizon line is visible where the sky 

In [5]:
print(type(model))

<class 'transformers_modules.lmms_hyphen_lab.LLaVA_hyphen_OneVision_hyphen_1_dot_5_hyphen_8B_hyphen_Instruct.bdf95183e0f91549cc71820b12d920b1a63689a2.modeling_llavaonevision1_5.LLaVAOneVision1_5_ForConditionalGeneration'>


In [5]:
def get_answer(image_path, query):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image_path,
                },
                {"type": "text", "text": query},
            ],
        }
    ]

    # Preparation for inference
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to("cuda")
    
    # Inference: Generation of the output
    generated_ids = model.generate(**inputs, max_new_tokens=1024)
    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )

    return output_text

# begin bird code

In [6]:
def get_bird_images(images_folder):
# Path to the folder containing bird subfolders
    # images_folder = '/projects/abbott_lab/Users/ishtiaque/datasets/CUB_200_2011/images'
    
    # Dictionary to store bird names and their corresponding image paths
    bird_images = {}
    
    # Iterate over each subfolder in the images folder
    for folder in os.listdir(images_folder):
        bird_name = folder.split(".")[-1]  # Extract bird name from folder name
        folder_path = os.path.join(images_folder, folder)  # Path to the bird's folder
        
        # Initialize an empty list to store image paths for the current bird
        image_paths = []
        
        # Iterate over the image files in the bird's folder
        for image_file in os.listdir(folder_path):
            image_path = os.path.join(folder_path, image_file)  # Full path to the image file
            image_paths.append(image_path)  # Store the image path
        
        # Store the list of image paths in the dictionary under the bird's name
        bird_images[bird_name] = image_paths
    
    # Now bird_images contains a dictionary where the keys are bird names and the values are lists of image paths
    print(len(bird_images))
    return bird_images



In [8]:
import locale, sys
print(sys.getdefaultencoding())
print(locale.getpreferredencoding(False))


utf-8
UTF-8


In [7]:
def get_json_data(json_file_name):

    # negated_questions, modified_new_cub_class_descriptions_full_fake, #modified_new_cub_class_descriptionsx #modified_mcqs_description_only2, modified_mcqs_description_only

    # For Task 1 type 1: 
    
    with open(json_file_name, "r") as file: 
        json_data = json.load(file)
    print(len(json_data))
    return json_data






In [8]:
def get_medium_hard_data(json_data):
    
# Use this if the json file contain the medium and hard categories
    medium_data = []
    hard_data = []

    data_counter = 0
    use_partial = False
    if use_partial:
        print("using limited data for debugging")
    
    for data in json_data:
        if data['difficulty'] == "Medium":
            medium_data.append(data)
        else:
            hard_data.append(data)

        

        data_counter = data_counter + 1
        if (data_counter>50) and use_partial:
            print(f"stopping at data = {data_counter}")
            break
        
    print(len(medium_data))
    print(len(hard_data))
    return medium_data, hard_data
    

In [13]:
def run_eval(data_partition):

    # with open(output_file_name, "a", encoding="utf-8") as f:
        # print (f"\n----------New file----------\n", file=f)


    if ("task_1a" in json_file_name): #correct ClassName
        print("task_1a\n")
    
    
    # Counters for distribution
    true_distribution = Counter()
    predicted_distribution = Counter()
    
    results = []
    bad_ans = 0
    
    for i, item in tqdm(enumerate(data_partition)): # for easy part json_data, for medium_data, for hard_data 
        mcq_id = item['mcq_id']
        question = item['question']
        options = item['options']
        correct_answer = item['correct_answer']
    
        if mcq_id not in bird_images or not bird_images[mcq_id]:
            print(f"No image for {mcq_id}") 
            continue
    
        image_paths = bird_images[mcq_id][:4]
    
        # Format the prompt
        # formatted_prompt = f"{question}\n"

        if ("task_1a" in json_file_name): #correct ClassName
            # print("task_1a\n")
            formatted_prompt = f"{question} Ignore the descriptions and focus only on the class names.\n"
        elif ("task_1b" in json_file_name): #correct Description
            formatted_prompt = f"{question} Ignore the class names and focus only on the descriptions.\n"
        else:
            formatted_prompt = f"{question}\n"

        # formatted_prompt = f"{question} Ignore the class names and focus on the descriptions.\n" # 
        for k in ['A', 'B', 'C', 'D']:  # ['D', 'C', 'B', 'A'] for position bias checking ['A', 'B', 'C', 'D']
            formatted_prompt += f"{k}. {options[k]}\n"
    
        # Final prompt
        prompt = f""" Your answer or response must ONLY be a single index ('A', 'B', 'C', 'D'). Do not response with any other text. 
    
        {formatted_prompt}
    
        Answer: ('A', 'B', 'C', 'D')"""
    
        # Run the model
        for image_path in image_paths:
            model_output = get_answer(image_path, prompt)
            # print("Model Output: ", model_output)
    
            # Extract predicted answer (basic string search, can refine)
            found_at_least_one = 0
            predicted_answer = None
            for option in ['A', 'B', 'C', 'D']:
                if f"{option}" in model_output or f"{option}." in model_output:
                    predicted_answer = option
                    found_at_least_one = 1
                    break
    
            # Update counters
            true_distribution[correct_answer] += 1
            if predicted_answer:
                predicted_distribution[predicted_answer] += 1

            if found_at_least_one == 0:
                bad_ans = bad_ans + 1
                print (f"Model not following output format!! {model_output}")
            
            results.append({
                'mcq_id': mcq_id,
                'image_path': image_path,
                'prompt': prompt,
                'model_output': model_output,
                'predicted_answer': predicted_answer,
                'correct_answer': correct_answer,
                'is_correct': predicted_answer == correct_answer
            })
    
    print(f"Results for file: {json_file_name}")
    # Accuracy summary 
    correct = sum(r['is_correct'] for r in results if r['predicted_answer'] is not None)
    total = len(results)
    print(f"Accuracy: {correct}/{total} = {correct / total:.2%}") 
    
    # Print distributions
    print("True Option Distribution:", dict(true_distribution))
    print("Predicted Option Distribution:", dict(predicted_distribution)) 

    with open(output_file_name, "a", encoding="utf-8") as f:
        print(f"\n ------- Results for file: {json_file_name}", file=f)
        print(f"Accuracy: {correct}/{total} = {correct / total:.2%}", file=f)
        print(f"bad_ans: {bad_ans}/{total} = {bad_ans / total:.2%}", file=f)

In [10]:
print("begin running code")


begin running code


In [14]:
print ("running for 4 images only!!!!!!!!!!!!!")
# json_files_list = ['negated_questions', 'new_cub_class_descriptions','new_cub_with_class_descriptions', 'modified_new_cub_class_descriptions_full_fake', 'modified_new_cub_class_descriptions_partial_fake', 'incorrect_class_correct_description', 'correct_class_incorrect_description']
output_file_name = "llava_ov_task1_eval_4images_log.txt"
bird_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/CUB_200_2011/images"
food_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/food-101/food-101/images"
aircraft_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/fgvc-aircraft-2013b/data/test"
dogs_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-dogs/images/Images"
car_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-cars/train"

bird_list = [
    "new_cub_class_descriptions",
    "new_cub_class_descriptions_task_0a_with_class_baseline",
    "new_cub_class_descriptions_task_0b_class_only_baseline",
    "new_cub_class_descriptions_task_1a",
    "new_cub_class_descriptions_task_1b",
    "new_cub_class_descriptions_task_2_full_fake",
]

food_list = [
    "new_food_class_descriptions",
    "new_food_class_descriptions_task_0a_with_class_baseline",
    "new_food_class_descriptions_task_0b_class_only_baseline",
    "new_food_class_descriptions_task_1a",
    "new_food_class_descriptions_task_1b",
    "new_food_class_descriptions_task_2_full_fake",
]

aircraft_list = [
    "new_aircraft_class_descriptions",
    "new_aircraft_class_descriptions_task_0a_with_class_baseline",
    "new_aircraft_class_descriptions_task_0b_class_only_baseline",
    "new_aircraft_class_descriptions_task_1a",
    "new_aircraft_class_descriptions_task_1b",
    "new_aircraft_class_descriptions_task_2_full_fake",
]

dogs_list = [
    "new_dogs_class_descriptions",
    "new_dogs_class_descriptions_task_0a_with_class_baseline",
    "new_dogs_class_descriptions_task_0b_class_only_baseline",
    "new_dogs_class_descriptions_task_1a",
    "new_dogs_class_descriptions_task_1b",
    "new_dogs_class_descriptions_task_2_full_fake",
]

car_list = [
    "new_car_class_descriptions",
    "new_car_class_descriptions_task_0a_with_class_baseline",
    "new_car_class_descriptions_task_0b_class_only_baseline",
    "new_car_class_descriptions_task_1a",
    "new_car_class_descriptions_task_1b",
    "new_car_class_descriptions_task_2_full_fake",
]

# json_files_list = food_list  #################### change!!!!!!!!!!!!!!!!!!!!
all_lists = [bird_list, food_list, aircraft_list, dogs_list, car_list]
folder_lists = [bird_folder, food_folder, aircraft_folder, dogs_folder, car_folder]

# all_lists = [food_list, aircraft_list, dogs_list, car_list]
# folder_lists = [food_folder, aircraft_folder, dogs_folder, car_folder]


for json_files_list, images_folder in zip(all_lists, folder_lists):

    bird_images = get_bird_images(images_folder)
    
    for json_file_name in json_files_list:
        
        json_file_name = f"/home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/{json_file_name}.json"
        json_data = get_json_data(json_file_name)
    
        # json_data[0]
        # json_data[1]
    
        medium_data, hard_data = get_medium_hard_data(json_data)
        
        # print("\n----Medium----")
        # run_eval(medium_data)
        print("\n----Hard----")
        run_eval(hard_data)
    

running for 4 images only!!!!!!!!!!!!!
200
400
200
200

----Hard----


200it [01:26,  2.30it/s]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_cub_class_descriptions.json
Accuracy: 503/800 = 62.88%
True Option Distribution: {'D': 220, 'A': 208, 'B': 172, 'C': 200}
Predicted Option Distribution: {'D': 156, 'C': 201, 'B': 233, 'A': 210}
400
200
200

----Hard----


200it [01:26,  2.31it/s]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_cub_class_descriptions_task_0a_with_class_baseline.json
Accuracy: 532/800 = 66.50%
True Option Distribution: {'D': 220, 'A': 208, 'B': 172, 'C': 200}
Predicted Option Distribution: {'D': 126, 'B': 259, 'C': 195, 'A': 220}
400
200
200

----Hard----


...

Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_car_class_descriptions_task_1a.json
Accuracy: 210/784 = 26.79%
True Option Distribution: {'A': 196, 'D': 200, 'B': 196, 'C': 192}
Predicted Option Distribution: {'C': 155, 'D': 81, 'A': 186, 'B': 362}
392
196
196

----Hard----


196it [02:16,  1.44it/s]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_car_class_descriptions_task_1b.json
Accuracy: 296/784 = 37.76%
True Option Distribution: {'A': 196, 'D': 200, 'B': 196, 'C': 192}
Predicted Option Distribution: {'A': 211, 'C': 184, 'D': 73, 'B': 316}
392
196
196

----Hard----


196it [02:14,  1.46it/s]

Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_car_class_descriptions_task_2_full_fake.json
Accuracy: 484/784 = 61.73%
True Option Distribution: {'A': 196, 'D': 200, 'B': 196, 'C': 192}
Predicted Option Distribution: {'C': 241, 'D': 146, 'B': 266, 'A': 131}


In [13]:
# print ("running for 4 images only!!!!!!!!!!!!!")
# json_files_list = ['negated_questions', 'new_cub_class_descriptions','new_cub_with_class_descriptions', 'modified_new_cub_class_descriptions_full_fake', 'modified_new_cub_class_descriptions_partial_fake', 'incorrect_class_correct_description', 'correct_class_incorrect_description']

bird_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/CUB_200_2011/images"
food_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/food-101/food-101/images"
aircraft_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/fgvc-aircraft-2013b/data/test"
dogs_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-dogs/images/Images"
car_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-cars/train"

bird_list = [
    "new_cub_class_descriptions_task_1b",
    "new_cub_class_descriptions_task_1a",
]

food_list = [
    "new_food_class_descriptions_task_1a",
    "new_food_class_descriptions_task_1b",
]

aircraft_list = [
    "new_aircraft_class_descriptions_task_1a",
    "new_aircraft_class_descriptions_task_1b",
]

dogs_list = [
    "new_dogs_class_descriptions_task_1a",
    "new_dogs_class_descriptions_task_1b",
]

car_list = [
    "new_car_class_descriptions_task_1a",
    "new_car_class_descriptions_task_1b",
]

# json_files_list = food_list  #################### change!!!!!!!!!!!!!!!!!!!!
all_lists = [bird_list, food_list, aircraft_list, dogs_list, car_list]
folder_lists = [bird_folder, food_folder, aircraft_folder, dogs_folder, car_folder]

# all_lists = [food_list, aircraft_list, dogs_list, car_list]
# folder_lists = [food_folder, aircraft_folder, dogs_folder, car_folder]


for json_files_list, images_folder in zip(all_lists, folder_lists):

    bird_images = get_bird_images(images_folder)
    
    for json_file_name in json_files_list:
        
        json_file_name = f"/home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/{json_file_name}.json"
        json_data = get_json_data(json_file_name)
    
        json_data[0]
        # json_data[1]
    
        medium_data, hard_data = get_medium_hard_data(json_data)
        
        print("\n----Medium----")
        run_eval(medium_data)
        print("\n----Hard----")
        run_eval(hard_data)
    

200
400
200
200

----Medium----


200it [02:12,  1.51it/s]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_cub_class_descriptions_task_1b.json
Accuracy: 935/1000 = 93.50%
True Option Distribution: {'D': 245, 'C': 220, 'B': 235, 'A': 300}
Predicted Option Distribution: {'D': 219, 'C': 218, 'B': 239, 'A': 324}

----Hard----


200it [02:11,  1.52it/s]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_cub_class_descriptions_task_1b.json
Accuracy: 256/1000 = 25.60%
True Option Distribution: {'D': 275, 'A': 260, 'B': 215, 'C': 250}
Predicted Option Distribution: {'D': 78, 'B': 387, 'A': 326, 'C': 209}
400
200
200

----Medium----
task_1a



...

Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_car_class_descriptions_task_1a.json
Accuracy: 264/980 = 26.94%
True Option Distribution: {'A': 245, 'D': 250, 'B': 245, 'C': 240}
Predicted Option Distribution: {'C': 187, 'D': 101, 'A': 248, 'B': 444}
392
196
196

----Medium----


196it [03:16,  1.00s/it]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_car_class_descriptions_task_1b.json
Accuracy: 912/980 = 93.06%
True Option Distribution: {'B': 250, 'C': 300, 'D': 250, 'A': 180}
Predicted Option Distribution: {'B': 242, 'C': 308, 'A': 205, 'D': 225}

----Hard----


196it [03:17,  1.01s/it]

Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_car_class_descriptions_task_1b.json
Accuracy: 370/980 = 37.76%
True Option Distribution: {'A': 245, 'D': 250, 'B': 245, 'C': 240}
Predicted Option Distribution: {'A': 266, 'C': 230, 'D': 94, 'B': 390}


# Cub-only

In [14]:
import json 
import package_name
print(package_name.__version__)

json_file_name = "/home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/task_3_negated_questions_bird.json"
with open(json_file_name, "r") as file: 
    json_data = json.load(file)

ModuleNotFoundError: No module named 'package_name'